# 10 · Agregados de dimensiones, indicador y clasificación

Reemplaza el antiguo inventario/EDA. Ejecute después del notebook 09. Lee el **mismo lote** y exige resultado completado del modelo 2.0; no recalcula D1, D2 ni D3.

Entrega agregados globales, por año, entidad, proveedor, departamento y tipo de contrato; porcentajes con denominadores explícitos, combinación de señales y resumen entidad-año de D3. Las tablas quedan en el notebook. No se exportan informes.


In [ ]:
from pathlib import Path
import json, sys, tempfile
import pandas as pd
import duckdb
from IPython.display import display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'secop_pipeline').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Abra el notebook dentro del proyecto.')
LOTE = 'desde_gold_v2_20261009'  # Debe coincidir con el notebook 09.
TOP = 30  # Límite de visualización; los porcentajes usan TODOS los contratos del grupo.
reportes = sorted((ROOT/'salida/alertas'/('lote_'+LOTE)).glob('*/ejecucion.json'),
                  key=lambda p:p.stat().st_mtime_ns, reverse=True)
if not reportes:
    raise RuntimeError('Ejecute primero el notebook 09 con este lote.')
manifest = reportes[0]
reporte = json.loads(manifest.read_text(encoding='utf-8'))
if reporte.get('estado') != 'COMPLETADA' or reporte.get('parametros',{}).get('version') != 'alertas-cd-2.0':
    raise RuntimeError('El último cálculo de este lote no está completo en versión 2.0. Consulte el notebook 11.')
work = manifest.parent
parquet = work/'resultados.parquet'
if not parquet.is_file():
    raise FileNotFoundError(parquet)
resumen = reporte['resumen']
print('Lote:', LOTE, '\nDirectorio:', work, '\nCorte:', reporte['parametros']['cutoff'])


## 1. Agregados globales disponibles sin leer contratos

Los conteos vienen del reporte validado. Se muestran contratos únicos, señales y clasificación por separado para evitar doble conteo. El porcentaje de activación por dimensión excluye sus no evaluables del denominador.


In [ ]:
display(pd.DataFrame.from_dict(resumen['dimensiones'], orient='index'))
total = resumen['contratos_resultado']
clases = pd.Series(resumen['clasificaciones'], name='contratos').to_frame()
clases['porcentaje_universo'] = 100*clases.contratos/total if total else float('nan')
display(clases)
display(pd.Series({k:resumen.get(k) for k in ['contratos_resultado','contratos_con_alguna_senal',
    'total_senales','contratos_evaluados_completos','contratos_informacion_insuficiente']},name='Cantidad').to_frame())
display(pd.DataFrame(resumen.get('combinaciones', [])))
display(pd.Series(resumen.get('coincidencias',{}),name='Contratos (pares incluyen triple)').to_frame())


## 2. Una lectura para todos los agregados

DuckDB agrupa columnas seleccionadas del Parquet mediante `GROUPING SETS`: global, año, entidad, proveedor, departamento y tipo. Conserva los faltantes como grupo explícito. Se limita la memoria a 1 GiB y dos hilos; usa archivos temporales si necesita desbordar y los elimina al cerrar la conexión al final.

**Medidas:** contratos, clasificación, señales por dimensión, denominadores evaluables, valor positivo válido y cobertura. El monto no incluye valores nulos, no finitos o ≤ 0; se informa cuántos se excluyeron. Los porcentajes del grupo se calculan desde sumas y conteos, nunca promediando porcentajes de subgrupos.

Los proveedores o entidades sin clave se agrupan como «sin dato», no se consideran una única organización identificada. D3 también se resume por entidad-año en la sección final.


In [ ]:
# Reejecutar esta celda cierra los temporales anteriores antes de volver a agregar.
if 'con' in globals():
    con.close()
if 'temporales' in globals():
    temporales.cleanup()
temporales = tempfile.TemporaryDirectory(prefix='secop_agregados_')
con = duckdb.connect()
def lit(s):
    return "'"+str(s).replace("'", "''")+"'"
con.execute("SET memory_limit='1GB'")
con.execute('SET threads=2')
con.execute('SET temp_directory='+lit(temporales.name))
con.execute(f"""CREATE TEMP TABLE agregados AS
WITH datos AS (
 SELECT year(firma) AS anio, entidad_clave, proveedor_clave, departamento, tipo_normalizado,
        d1_concentracion AS d1, d2_valor_atipico AS d2, alerta_d3 AS d3,
        score_min, score_max, dimensiones_evaluadas, clasificacion,
        CASE WHEN isfinite(valor_contrato) AND valor_contrato>0 THEN valor_contrato END AS valor
 FROM read_parquet({lit(parquet.as_posix())})
)
SELECT CASE WHEN grouping(anio)=0 THEN 'Año'
 WHEN grouping(entidad_clave)=0 THEN 'Entidad' WHEN grouping(proveedor_clave)=0 THEN 'Proveedor'
 WHEN grouping(departamento)=0 THEN 'Departamento' WHEN grouping(tipo_normalizado)=0 THEN 'Tipo'
 ELSE 'Global' END AS agregado,
 CASE WHEN grouping(anio)=0 THEN coalesce(cast(anio AS VARCHAR),'(sin dato)')
 WHEN grouping(entidad_clave)=0 THEN coalesce(entidad_clave,'(sin dato)')
 WHEN grouping(proveedor_clave)=0 THEN coalesce(proveedor_clave,'(sin dato)')
 WHEN grouping(departamento)=0 THEN coalesce(departamento,'(sin dato)')
 WHEN grouping(tipo_normalizado)=0 THEN coalesce(tipo_normalizado,'(sin dato)') ELSE 'Todos' END AS grupo,
 count(*) AS contratos,
 count(*) FILTER(WHERE clasificacion='ALERTA') AS clasificados_alerta,
 count(*) FILTER(WHERE clasificacion='NO_ALERTA') AS clasificados_sin_alerta,
 count(*) FILTER(WHERE clasificacion='NO_EVALUABLE') AS clasificados_no_evaluables,
 count(*) FILTER(WHERE d1=1 OR d2=1 OR d3=1) AS contratos_con_senal,
 count(*) FILTER(WHERE d1=1) AS alertas_d1, count(d1) AS evaluados_d1,
 count(*) FILTER(WHERE d2=1) AS alertas_d2, count(d2) AS evaluados_d2,
 count(*) FILTER(WHERE d3=1) AS alertas_d3, count(d3) AS evaluados_d3,
 100.0*sum(dimensiones_evaluadas)/nullif(3*count(*),0) AS cobertura_pct,
 100.0*sum(score_min)/nullif(3*count(*),0) AS criterios_activados_min_pct,
 100.0*sum(score_max)/nullif(3*count(*),0) AS criterios_activados_max_pct,
 count(valor) AS contratos_valor_valido, count(*)-count(valor) AS contratos_valor_excluido,
 sum(valor) AS valor_valido_total,
 sum(valor) FILTER(WHERE clasificacion='ALERTA') AS valor_valido_clasificado_alerta
FROM datos GROUP BY GROUPING SETS ((),(anio),(entidad_clave),(proveedor_clave),(departamento),(tipo_normalizado))
""")
con.execute("""CREATE TEMP VIEW porcentajes AS SELECT *,
 100.0*clasificados_alerta/nullif(contratos,0) AS clasificados_alerta_pct,
 100.0*clasificados_no_evaluables/nullif(contratos,0) AS clasificados_no_evaluables_pct,
 100.0*alertas_d1/nullif(evaluados_d1,0) AS activacion_d1_pct,
 100.0*alertas_d2/nullif(evaluados_d2,0) AS activacion_d2_pct,
 100.0*alertas_d3/nullif(evaluados_d3,0) AS activacion_d3_pct
FROM agregados""")
print('Agregados preparados. Las siguientes tablas reutilizan este cálculo.')


## 3. Tablas de agregados

Entidades y proveedores se ordenan por cantidad de contratos clasificados ALERTA, luego por volumen. Son grupos para priorizar revisión, no una clasificación de corrupción. Compare siempre tasa y cobertura: el número bruto depende del tamaño del grupo. Solo se muestran los primeros `TOP`; los agregados completos permanecen consultables en `porcentajes` durante esta sesión.


In [ ]:
tablas = {}
for nivel in ('Global','Año','Entidad','Proveedor','Departamento','Tipo'):
    orden = 'grupo' if nivel=='Año' else 'clasificados_alerta DESC, contratos DESC, grupo'
    limite = '' if nivel in ('Global','Año') else f' LIMIT {int(TOP)}'
    tablas[nivel] = con.execute('SELECT * FROM porcentajes WHERE agregado=? ORDER BY '+orden+limite,[nivel]).df()
    print('AGREGADO:', nivel)
    display(tablas[nivel])
globales = tablas['Global']
if not globales.empty and globales.iloc[0].contratos:
    r = globales.iloc[0]
    print(f'Conclusión global: {int(r.clasificados_alerta):,} contratos clasificados ALERTA '
          f'({r.clasificados_alerta_pct:.2f}% del universo). Cobertura: {r.cobertura_pct:.2f}%. '
          f'Activación media de criterios: entre {r.criterios_activados_min_pct:.2f}% y '
          f'{r.criterios_activados_max_pct:.2f}%. El rango refleja dimensiones desconocidas.')


## 4. D3 por entidad-año

Una entidad-año puede producir muchos contratos de diciembre señalados. Esta tabla cuenta una sola vez cada entidad-año, separa cobertura anual y calcula la tasa sobre entidades-año evaluables. Lee el resumen pequeño guardado por D3.


In [ ]:
d3_path = work/'d3_entidad_anio.parquet'
if not d3_path.is_file():
    print('Falta el resumen pequeño D3. Reanude el notebook 09 para regenerarlo.')
else:
    d3_anual = con.execute(f"""SELECT d3_anio AS anio, count(*) AS entidades_anio,
        count(*) FILTER(WHERE motivo_entidad_anio_d3 IS NULL) AS evaluables,
        count(*) FILTER(WHERE motivo_entidad_anio_d3 IS NOT NULL) AS no_evaluables,
        count(*) FILTER(WHERE motivo_entidad_anio_d3 IS NULL AND participacion_diciembre>umbral_d3) AS alertadas,
        100.0*count(*) FILTER(WHERE motivo_entidad_anio_d3 IS NULL AND participacion_diciembre>umbral_d3)
          /nullif(count(*) FILTER(WHERE motivo_entidad_anio_d3 IS NULL),0) AS activacion_pct
        FROM read_parquet({lit(d3_path.as_posix())}) GROUP BY d3_anio ORDER BY d3_anio""").df()
    display(d3_anual)


## 5. Cerrar recursos

Ejecute esta celda al terminar. Las tablas mostradas y los DataFrames `tablas` y `d3_anual` siguen en el notebook. Para consultar otros grupos después de cerrar, vuelva a ejecutar la sección 2.


In [ ]:
con.close()
temporales.cleanup()
print('Conexión y archivos temporales cerrados. Resultados mostrados en el notebook.')
